In [ ]:
import numpy as np
import sys
import os
import nibabel as nib
sys.path.append(os.getcwd().removesuffix('\\evaluation'))
print(sys.path)

from models.tooth_fairy_1 import build_hyspark_tooth_fairy_1
from models.class_segmentor_model import build_class_segmentor_model
import torch as tc
import torch.nn as nn
from evaluation.evaluate_agh_segmentor import evaluate_agh

model =build_class_segmentor_model(inchannels=1, outchannels=34, depths=[16, 32, 64, 128, 192])
model.load_state_dict(tc.load('../models/weights/agh_segmentor_best.pt', weights_only=True))
file = '../data/snippets_img/1.npy'

image = np.load(file).astype(np.float32)

print(image.shape)

model = model.cuda()

image = tc.tensor(image[None, ...], dtype=tc.float32).cuda()

output = nn.Sigmoid()(model(image))

output[output > 0.5] = 1
output[output < 0.5] = 0

output = output.detach().cpu().numpy().astype(np.uint8)

affine = np.eye(4)
nifti_img = nib.Nifti1Image(output, affine)
nib.save(nifti_img, 'evaluation/evaluation_results/output_image.nii.gz')

In [ ]:
import numpy as np
import sys
import os
import nibabel as nib
sys.path.append(os.getcwd().removesuffix('\\evaluation'))
print(sys.path)

from models.tooth_fairy_1 import build_hyspark_tooth_fairy_1
from models.class_segmentor_model import build_class_segmentor_model
import torch as tc
import torch.nn as nn
from evaluation.evaluate_agh_segmentor import evaluate_agh
from processing.utils import load_itk, crop_to_mask
from processing.utils import embed_in_center



model =build_hyspark_tooth_fairy_1(in_channel=1, n_classes=1, img_size=96, freeze=False)
model.load_state_dict(tc.load('../models/weights/tooth_fairy_3d_best.pt', weights_only=True))
file = 'D:/db1_v1.0/db1_v1.0_nifti/10/image.nii.gz'
helper = 'D:/db1_v1.0/db1_v1.0_nifti_teeth_masks_multi_cls_ordered_weak/10.nii.gz'

image = load_itk(file)
helper = load_itk(helper)

model = model.cuda()

output = np.zeros(image.shape)

for teeth_num in np.unique(helper)[1:]:

    temp_mask = np.zeros(helper.shape)

    temp_mask[helper != teeth_num] = 0
    temp_mask[helper == teeth_num] = 1 

    image_snipped, mask_snipped, coords = crop_to_mask(image, temp_mask, padding=5, input=1, coords=True)
    
    masked = image_snipped[mask_snipped>0]
    image_snipped[mask_snipped>0] = (masked - masked.min())/(masked.max() - masked.min() + 1e-5)
    image_snipped[mask_snipped<1] = 0 

    if image_snipped.shape[0] < 96 and image_snipped.shape[1] < 96 and image.shape[2] < 96: 
            image_snip_2, cords_2 = embed_in_center(image_snipped, 96, coords=True)
            mask_snip_2 = embed_in_center(mask_snipped, 96)
        
    else:   
            image_snip_2, cords_2 = embed_in_center(image_snipped, 2*96, coords=True)
            mask_snip_2 = embed_in_center(mask_snipped, 2*96)

            image_snip_2 = image_snip_2[:, 96//2:96//2+96, 96//2:96//2+96]
            mask_snip_2 = mask_snip_2[:, 96//2:96//2+96, 96//2:96//2+96]

            image_snip_2_new = np.zeros((2, 96, 96, 96))
            mask_snip_2_new = np.zeros((2, 96, 96, 96))

            image_snip_2_new[0] = image_snip_2[:96]
            image_snip_2_new[1] = image_snip_2[96:]

            mask_snip_2_new[0] = mask_snip_2[:96]
            mask_snip_2_new[1] = mask_snip_2[96:]

image = tc.tensor(image[None, ...], dtype=tc.float32).cuda()

output = nn.Sigmoid()(model(image))

output[output > 0.5] = 1
output[output < 0.5] = 0

output = output.detach().cpu().numpy().astype(np.uint8)

affine = np.eye(4)
nifti_img = nib.Nifti1Image(output, affine)
nib.save(nifti_img, 'evaluation/evaluation_results/output_image.nii.gz')